# 06 · Section embeddings and year-over-year narrative drift

**Reads** `stg_clean_10Ks` (dbt) · **Writes** `fact_section_embeddings`, `fact_section_drift`, `fact_risk_drift`

*Drift* = cosine distance between a section's embedding and the same section in the company's previous filing
(0 = same story, larger = more change). The three copy-pasted drift blocks of the old notebook are now one loop over
sections; `fact_risk_drift` (RISKS only) is kept because the dbt mart reads it. Distances are rounded to 4 decimals
(the old 2-decimal rounding erased most of the variation).

The OpenAI key is read from a Databricks secret scope (`openai` / `openai-api-key`).

In [ ]:
%pip install openai

In [ ]:
%restart_python

In [ ]:
# Make the repository's Python package importable (Databricks Git folder: cwd is this notebook's folder)
import os, sys
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from trading_sentiment import table

In [ ]:
from pyspark.sql import functions as F

EMBED_MODEL = "text-embedding-3-large"
MAX_CHARS = 20_000            # about 5k tokens, under the model's 8,191-token input limit
api_key = dbutils.secrets.get("openai", "openai-api-key")

sections = (spark.table(table("stg_clean_10Ks"))
    .selectExpr("cik", "company_name", "tickers", "filing_date", "accessionNumber", "market_cap", "sector",
                "stack(3, 'BUSINESS', bus_text, 'RISKS', risks_text, 'MDA', mda_text) as (section_name, section_text)")
    .where("section_text is not null and length(trim(section_text)) > 2000")
    .withColumn("section_text_trimmed", F.substring("section_text", 1, MAX_CHARS)))

## 1 · Embed

In [ ]:
from pyspark.sql.types import ArrayType, DoubleType

_client = None

def embed_text(text):
    """One OpenAI client per Python worker process (the old code created a client for every row)."""
    global _client
    if not text or not text.strip():
        return None
    if _client is None:
        from openai import OpenAI
        _client = OpenAI(api_key=api_key)                  # the SDK retries rate-limit errors itself
    return _client.embeddings.create(model=EMBED_MODEL, input=text).data[0].embedding

embed_udf = F.udf(embed_text, ArrayType(DoubleType()))
embedded = sections.withColumn("embedding", embed_udf("section_text_trimmed")).drop("section_text", "section_text_trimmed")
embedded.write.format("delta").mode("overwrite").saveAsTable(table("fact_section_embeddings"))

## 2 · Drift versus the previous filing, for every section

In [ ]:
import numpy as np
from pyspark.sql.types import DoubleType
from pyspark.sql.window import Window

def cosine_distance(a, b):
    if a is None or b is None:
        return None
    a, b = np.asarray(a), np.asarray(b)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(1.0 - np.dot(a, b) / denom) if denom else None

cosine_udf = F.udf(cosine_distance, DoubleType())

emb = spark.table(table("fact_section_embeddings"))
w = Window.partitionBy("cik", "section_name").orderBy(F.desc("filing_date"))     # previous filing = next row

drift = (emb
    .withColumn("prev_embedding", F.lead("embedding").over(w))
    .withColumn("prev_filing_date", F.lead("filing_date").over(w))
    .where("prev_embedding is not null")
    .withColumn("cosine_distance", F.round(cosine_udf("embedding", "prev_embedding"), 4))
    .select("cik", "company_name", "tickers", "filing_date", "prev_filing_date", "section_name", "cosine_distance"))

drift.write.format("delta").mode("overwrite").saveAsTable(table("fact_section_drift"))

# table read by dbt (comp_kpi_filing)
(drift.where("section_name = 'RISKS'")
      .withColumnRenamed("cosine_distance", "risk_cosine_distance")
      .drop("section_name")
      .write.format("delta").mode("overwrite").saveAsTable(table("fact_risk_drift")))

display(drift.where("cosine_distance >= 0.2").orderBy(F.desc("cosine_distance")).limit(20))